# Preparação dos dados selecionados — cheias do Guaíba de 2024

Este notebook é destinado à avaliação e eventual preparação das bases selecionadas a partir da [exploração inicial](processamento_cheias_guaiba_2024.ipynb).

O foco compreende bairros, escolas, SUS, atenção primária e camadas de inundação em Porto Alegre. Os procedimentos de limpeza e validação ainda dependem da identificação dos problemas e da definição dos tratamentos. Nenhuma transformação foi realizada nesta estrutura inicial.

Os arquivos de `dados/brutos/` devem ser preservados. Eventuais produtos preparados poderão ser destinados a `dados/tratados/`, mantendo as informações geoespaciais relevantes.


## 1. Bibliotecas e localização dos arquivos

Pandas e GeoPandas apoiam a preparação tabular e espacial; Pyogrio, a leitura do GeoPackage; Shapely, as operações geométricas; e PyArrow, a exportação em GeoParquet. As bibliotecas padrão `pathlib`, `json` e `sqlite3` permitem trabalhar com caminhos, cenários e consultas ao banco.

Os caminhos pressupõem execução a partir de `notebooks/`, utilizando seu diretório pai como raiz do projeto. O JSON do INCT está em `dados/data.json`. Nenhuma base é carregada ou alterada nesta etapa.

In [1]:
from pathlib import Path
import json
import sqlite3

import pandas as pd
import geopandas as gpd
import pyogrio
import shapely
import pyarrow

In [2]:
RAIZ = Path.cwd().resolve().parents[0]

DADOS = RAIZ / "dados"
DADOS_BRUTOS = DADOS / "brutos"
DADOS_TRATADOS = DADOS / "tratados"

ARQUIVO_BANCO = DADOS_BRUTOS / "cheias_rhguaiba_2024_db_v5.gpkg"
ARQUIVO_GLOSSARIO = DADOS / "cheias_rhguaiba_2024_db_v5_glossario.csv"
ARQUIVO_JSON_INCT = DADOS / "data.json"

arquivos_entrada = {
    "GeoPackage": ARQUIVO_BANCO,
    "Glossário": ARQUIVO_GLOSSARIO,
    "JSON do INCT": ARQUIVO_JSON_INCT,
}

for nome, caminho in arquivos_entrada.items():
    if not caminho.is_file():
        raise FileNotFoundError(f"{nome} não encontrado: {caminho}")
    print(f"{nome}: {caminho}")

print(f"Diretório de saída: {DADOS_TRATADOS}")

GeoPackage: /home/lucas/CDIA/CPA/ProjetoExtensionista-CPA/dados/brutos/cheias_rhguaiba_2024_db_v5.gpkg
Glossário: /home/lucas/CDIA/CPA/ProjetoExtensionista-CPA/dados/cheias_rhguaiba_2024_db_v5_glossario.csv
JSON do INCT: /home/lucas/CDIA/CPA/ProjetoExtensionista-CPA/dados/data.json
Diretório de saída: /home/lucas/CDIA/CPA/ProjetoExtensionista-CPA/dados/tratados


## 2. Referência territorial — bairros

**Camada:** `poa_admin_bairros`.

Possíveis verificações: validade e cobertura das geometrias, consistência dos identificadores e correspondência entre os 83 registros territoriais e os 94 bairros presentes nos cenários do JSON do INCT.


### Leitura da camada

Carregamos somente `poa_admin_bairros`. As verificações seguintes não corrigem ou excluem registros.

In [3]:
bairros = gpd.read_file(
    ARQUIVO_BANCO,
    layer="poa_admin_bairros"
)

display(bairros)

,nmbop,cdbop,nmrop,cdrop,nmrp,cdrp,cota_mean,geometry
0,AGRONOMIA,BOP71,LOMBA DO PINHEIRO,OP04,LOMBA DO PINHEIRO - PARTENON,RP07,125.157306,"MULTIPOLYGON Z (((489305.255 6676873.088 0, 48..."
1,ANCHIETA,BOP19,HUMAITA-NAVEGANTES,OP01,HUMAITA - NAVEGANTES - ILHAS E NOROESTE,RP02,1.954820,"MULTIPOLYGON Z (((484961.018 6685092.233 0, 48..."
2,ARQUIPELAGO,BOP36,ILHAS,OP17,HUMAITA - NAVEGANTES - ILHAS E NOROESTE,RP02,1.162602,"MULTIPOLYGON Z (((471983.597 6674237.441 0, 47..."
3,AUXILIADORA,BOP01,CENTRO,OP16,CENTRO,RP01,28.703022,"MULTIPOLYGON Z (((482207 6678761.422 0, 482243..."
4,AZENHA,BOP02,CENTRO,OP16,CENTRO,RP01,12.897710,"MULTIPOLYGON Z (((479842.718 6675588.541 0, 47..."
...,...,...,...,...,...,...,...,...
78,SANTA MARIA GORETTI,BOP32,NOROESTE,OP02,HUMAITA - NAVEGANTES - ILHAS E NOROESTE,RP02,4.145914,"MULTIPOLYGON Z (((483675.3 6680321.662 0, 4836..."
79,CAVALHADA,BOP57,CENTRO SUL,OP12,CENTRO-SUL E SUL,RP06,23.222729,"MULTIPOLYGON Z (((478348.363 6670374.305 0, 47..."
80,PRAIA DE BELAS,BOP15,CENTRO,OP16,CENTRO,RP01,2.401748,"MULTIPOLYGON Z (((477146.457 6677062.151 0, 47..."
81,PONTA GROSSA,BOP83,EXTREMO SUL,OP13,RESTINGA - EXTREMO SUL,RP08,14.849819,"MULTIPOLYGON Z (((482726.295 6662157.991 0, 48..."


### Identificadores e geometrias

Verificamos ausências e repetições nos nomes e códigos, além de geometrias nulas, vazias e inválidas. A validade geométrica não comprova precisão ou cobertura territorial.

In [4]:
print(f"Quantidade de registros: {len(bairros)}")
print(f"CRS: {bairros.crs}")

print("\nAusências nos identificadores:")
display(bairros[["nmbop", "cdbop"]].isna().sum())

print("\nNomes repetidos:")
display(bairros.loc[bairros["nmbop"].duplicated(keep=False)])

print("\nCódigos repetidos:")
display(bairros.loc[bairros["cdbop"].duplicated(keep=False)])

geometria_nula = bairros.geometry.isna()
geometria_vazia = bairros.geometry.is_empty
geometria_invalida = (
    ~geometria_nula
    & ~geometria_vazia
    & ~bairros.geometry.is_valid
)

print(f"\nGeometrias nulas: {geometria_nula.sum()}")
print(f"Geometrias vazias: {geometria_vazia.sum()}")
print(f"Geometrias inválidas não vazias: {geometria_invalida.sum()}")

display(bairros.loc[
    geometria_nula | geometria_vazia | geometria_invalida
])

Quantidade de registros: 83
CRS: EPSG:31982

Ausências nos identificadores:


nmbop    0
cdbop    0
dtype: int64


Nomes repetidos:


,nmbop,cdbop,nmrop,cdrop,nmrp,cdrp,cota_mean,geometry



Códigos repetidos:


,nmbop,cdbop,nmrop,cdrop,nmrp,cdrp,cota_mean,geometry



Geometrias nulas: 0
Geometrias vazias: 0
Geometrias inválidas não vazias: 0


,nmbop,cdbop,nmrop,cdrop,nmrp,cdrp,cota_mean,geometry


### Nomes dos bairros nos cenários do INCT

Usamos o primeiro cenário como referência e comparamos seus nomes com os demais. Diferenças são apresentadas para investigação, sem alterar o JSON.

In [5]:
with ARQUIVO_JSON_INCT.open(encoding="utf-8") as arquivo:
    cenarios = json.load(arquivo)

primeiro_cenario = next(iter(cenarios))
nomes_json = set(cenarios[primeiro_cenario])

diferencas_entre_cenarios = {}

for nome_cenario, registros in cenarios.items():
    nomes_cenario = set(registros)

    if nomes_cenario != nomes_json:
        diferencas_entre_cenarios[nome_cenario] = {
            "somente_neste_cenario": sorted(nomes_cenario - nomes_json),
            "ausentes_neste_cenario": sorted(nomes_json - nomes_cenario),
        }

print(f"Cenário de referência: {primeiro_cenario}")
print(f"Bairros no cenário: {len(nomes_json)}")
print(f"Cenários com nomes diferentes: {len(diferencas_entre_cenarios)}")

display(diferencas_entre_cenarios)

Cenário de referência: 3_0m
Bairros no cenário: 94
Cenários com nomes diferentes: 0


{}

### Correspondência por nome e equivalências adotadas

As equivalências abaixo foram aceitas pela equipe para relacionar os nomes do GeoPackage aos do JSON, preservando `nmbop` e `cdbop` originais. A coluna auxiliar `nome_comparacao` incorpora essas correspondências antes da normalização de acentos, caixa e espaços.

| Nome no GeoPackage | Nome no JSON |
| --- | --- |
| CEL APARICIO BORGES | Coronel Aparício Borges |
| MONTSERRAT | Mont'Serrat |
| PASSO DAREIA | Passo D'Areia |
| SAO JOSE | Vila São José |

A correspondência de São José foi aceita com apoio na posição do polígono, que faz divisa com Partenon, e nos atributos que o associam às regiões PARTENON e LOMBA DO PINHEIRO – PARTENON. Isso não comprova igualdade dos limites territoriais adotados pelas duas fontes.

`left_only` indica nomes apenas no GeoPackage; `right_only`, apenas no JSON. Bairros restantes sem correspondência são mantidos para investigação, sem associação automática ou exclusão. Nenhum dado original é modificado e nenhuma tabela preparada é exportada nesta etapa.


In [6]:
import unicodedata


def normalizar_nome(nome):
    nome = unicodedata.normalize("NFKD", nome)
    nome = "".join(
        caractere
        for caractere in nome
        if not unicodedata.combining(caractere)
    )

    return " ".join(nome.casefold().split())


equivalencias_bairros = {
    "CEL APARICIO BORGES": "Coronel Aparício Borges",
    "MONTSERRAT": "Mont'Serrat",
    "PASSO DAREIA": "Passo D'Areia",
    "SAO JOSE": "Vila São José",
}

comparacao_gpkg = bairros[["nmbop", "cdbop"]].copy()
comparacao_gpkg["nome_comparacao"] = (
    comparacao_gpkg["nmbop"].replace(equivalencias_bairros)
)
comparacao_gpkg["chave_nome"] = (
    comparacao_gpkg["nome_comparacao"].map(
        lambda nome: normalizar_nome(nome) if pd.notna(nome) else None
    )
)

comparacao_json = pd.DataFrame({
    "bairro_json": sorted(nomes_json)
})
comparacao_json["chave_nome"] = (
    comparacao_json["bairro_json"].map(normalizar_nome)
)

comparacao = comparacao_gpkg.merge(
    comparacao_json,
    on="chave_nome",
    how="outer",
    indicator=True
)

print("Resumo da correspondência:")
display(comparacao["_merge"].value_counts())

print("\nNomes com chaves repetidas após normalização:")
display(comparacao_gpkg.loc[
    comparacao_gpkg["chave_nome"].duplicated(keep=False)
])
display(comparacao_json.loc[
    comparacao_json["chave_nome"].duplicated(keep=False)
])

print("\nBairros sem correspondência por nome:")
display(comparacao.loc[
    comparacao["_merge"] != "both",
    ["nmbop", "cdbop", "bairro_json", "_merge"]
])

Resumo da correspondência:


_merge
both          83
right_only    11
left_only      0
Name: count, dtype: int64


Nomes com chaves repetidas após normalização:


,nmbop,cdbop,nome_comparacao,chave_nome


,bairro_json,chave_nome



Bairros sem correspondência por nome:


,nmbop,cdbop,bairro_json,_merge
0,NaN,NaN,Aberta dos Morros,right_only
10,NaN,NaN,Boa Vista do Sul,right_only
22,NaN,NaN,Costa e Silva,right_only
26,NaN,NaN,Extrema,right_only
40,NaN,NaN,Jardim Europa,right_only
44,NaN,NaN,Jardim Leopoldina,right_only
59,NaN,NaN,Parque Santa Fé,right_only
65,NaN,NaN,Pitinga,right_only
73,NaN,NaN,Santa Rosa de Lima,right_only
77,NaN,NaN,São Caetano,right_only


## 3. Educação — localização e identificação das escolas

**Camada:** `poa_edu_escolas`.

O foco é identificação e localização. Possíveis verificações incluem os códigos INEP e as coordenadas compartilhadas.

Permanece o critério adotado na exploração: considerar os 1.062 registros pertencentes a Porto Alegre, mantendo as coordenadas originais e a sinalização dos cinco pontos externos à cobertura dos polígonos de bairros.


## 4. Saúde — cadastro SUS

**Camada:** `poa_saude_sus`.

Uma possível etapa é investigar localizações compartilhadas e esclarecer se os registros representam serviços, estabelecimentos ou endereços, antes de definir contagens ou tratamentos.


## 5. Atenção primária — versões temporais

**Camadas:** `poa_saude_unidades_atencao_primaria_09052024` e `poa_saude_unidades_atencao_primaria_10052024`.

Possíveis verificações: correspondência das unidades, inconsistência do campo `Cidade`, situações de atendimento e referência temporal. A escolha de versão depende da pergunta de análise; as versões não devem ser somadas diretamente ou combinadas automaticamente com o cadastro SUS.


## 6. Inundação em Porto Alegre — eventos e simulações

**Escopo selecionado:** seis camadas associadas a eventos pelo nome, de 1941, 1967, 2015, setembro e novembro de 2023 e 2024; e cinco cenários simulados nominais de 500, 550, 600, 650 e 700 cm.

As divergências entre nomes e descrições do [glossário](../dados/cheias_rhguaiba_2024_db_v5_glossario.csv), registradas na exploração inicial, precisam de esclarecimento antes de comparações quantitativas. Possíveis verificações adicionais incluem classes, geometrias, sobreposições, unidades de área e compatibilidade com os cenários do JSON.


## 7. Registro de decisões e validação dos resultados

Espaço para documentar cada problema identificado, a justificativa de eventual tratamento e seus efeitos. As verificações de resultado poderão ser definidas conforme as transformações escolhidas.


## 8. Possíveis produtos preparados

O formato escolhido para os futuros produtos geoespaciais é **GeoParquet (`.parquet`)**, com um arquivo por camada em `dados/tratados/`. Atributos, recortes e nomes de saída podem ser definidos conforme os tratamentos aprovados, preservando CRS e geometrias quando pertinentes. Nenhuma exportação foi realizada nesta etapa.